# 12. Query, Key and ValueFrom *Transformers from Scratch*.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or# three files the chapter uses. If you cloned the repository it# finds them where they already are and downloads nothing.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"PLACES = [".", "code", "../code"]for name in ["ch12_qkv.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Query, key and value, as a lookup that is allowed to blend.An ordinary dictionary matches a key exactly or fails. Attention scores every key, turns those scores into amounts, and returns a mixture. This file puts the two side by side.

In [ ]:
import numpy as npnp.set_printoptions(precision=3, suppress=True, floatmode="fixed")KEYS = np.array([[1.0, 0.0],                 [0.0, 1.0],                 [-1.0, 0.0],                 [0.0, -1.0]])VALUES = np.array([[10.0, 0.0],                   [0.0, 10.0],                   [-10.0, 0.0],                   [0.0, -10.0]])NAMES = ["north", "east", "south", "west"]

In [ ]:
def softmax(scores):    shifted = scores - scores.max()    e = np.exp(shifted)    return e / e.sum()

A dictionary. A key matches exactly, or nothing comes back.

In [ ]:
def exact_lookup(query):    for name, key, value in zip(NAMES, KEYS, VALUES):        if np.allclose(query, key):            return name, value    return None, None

Score every key, normalise, then blend the values by those.

In [ ]:
def soft_lookup(query, sharpness=4.0):    scores = KEYS @ query * sharpness    weights = softmax(scores)    return weights, weights @ VALUES

One input row becomes three different rows.

In [ ]:
def projections(x, w_q, w_k, w_v):    return x @ w_q, x @ w_k, x @ w_v

In [ ]:
def main():    print("AN ORDINARY LOOKUP")    for query in [np.array([1.0, 0.0]), np.array([0.7, 0.7])]:        name, value = exact_lookup(query)        found = f"{name}, {value}" if name else "nothing at all"        print(f"  query {query}  ->  {found}")    print("\nTHE SAME TWO QUERIES, SCORED AND BLENDED")    for query in [np.array([1.0, 0.0]), np.array([0.7, 0.7])]:        weights, blended = soft_lookup(query)        print(f"  query {query}")        for name, weight in zip(NAMES, weights):            print(f"    {name:<7} {weight:.3f}")        print(f"    result {blended}")    print("\nA QUERY THAT MATCHES NOTHING STILL GETS AN ANSWER")    weights, blended = soft_lookup(np.array([0.0, 0.0]))    print("  query [0. 0.]")    print("    weights", weights)    print("    result ", blended)    print("  every key scored nought, so the answer is an even blend")    print("\nONE INPUT, THREE DIFFERENT VECTORS")    x = np.array([1.0, 2.0])    w_q = np.array([[1.0, 0.0], [0.0, 1.0]])    w_k = np.array([[0.0, 1.0], [1.0, 0.0]])    w_v = np.array([[2.0, 0.0], [0.0, 0.5]])    q, k, v = projections(x, w_q, w_k, w_v)    print("  input   ", x)    print("  query   ", q, " what this position is looking for")    print("  key     ", k, " what it offers to be matched against")    print("  value   ", v, " what it contributes when attended to")    print("  three roles, three matrices, one input")

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.